### Step 1 - Investigate the Bronze Data

In [0]:
df_bronze = spark.table("workspace.data_analysis.bronze_employees")

In [0]:
df_bronze.printSchema()

root
 |-- Employee_ID: string (nullable = true)
 |-- First_Name: string (nullable = true)
 |-- Last_Name: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone_Number: string (nullable = true)
 |-- Department_Region: string (nullable = true)
 |-- Birth_Year: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- Join_Date: string (nullable = true)
 |-- Salary: string (nullable = true)
 |-- Performance_Score: string (nullable = true)
 |-- Remote_Work: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



In [0]:
display(df_bronze.limit(10))
print("Total rows:",df_bronze.count())

Employee_ID,First_Name,Last_Name,Email,Phone_Number,Department_Region,Birth_Year,Age,Join_Date,Salary,Performance_Score,Remote_Work,Status,batch_id,source_system,ingestion_timestamp
EMP3000,Ganesh,Reddy,ganesh.reddy@example.com,9542351161,Support-Karnataka,null,59,8/31/2020,100252.34,Poor,Yes,Inactive,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3001,Lakshmi,Nathan,lakshmi.nathan@example.com,8255341928,HR-Punjab,2004,22,2/26/2017,113071.48,Average,No,Pending,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3002,Meena,Joshi,meena.joshi@example.com,7884969653,Sales-Delhi,1985,41,3/29/2025,142863.71,Poor,1,Inactive,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3003,Rahul,Bose,rahul.bose@example.com,6462704828,Operations-Rajasthan,1989,37,10/26/2017,146384.8,Excelent,N,On Leave,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3004,Shalini,Verma,shalini.verma@example.com,7278248963,Support-Tamil Nadu,1996,null,11/8/2024,148165.17,Good,No,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3005,Kiran,Rao,kiran.rao@example.com,8738299737,DevOps-Karnataka,1969,57,7/31/2017,null,Excellent,Y,Inactive,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3006,Sanjay,Reddy,sanjay.reddy@example.com,6801326773,HR-Tamil Nadu,1986,40,2/23/2017,48776.11,Average,Yes,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3007,Prakash,Sharma,prakash.sharma@example.com,7191361939,Operations-Punjab,1996,30,11/11/2025,32310.41,Average,TRUE,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3008,Nisha,Iyer,nisha.iyer@example.com,8251354278,Marketing-Delhi,1973,53,7/19/2018,141127.29,null,No,Inactive,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3009,Nisha,null,nisha@example.com,6164005242,Admin-Karnataka,1973,53,4/29/2023,94152.96,Average,Y,On Leave,BATCH_001,employee_app,2026-09-26T12:44:46.516Z


Total rows: 1000


In [0]:
from pyspark.sql.functions import col,trim,when,sum

null_summary = df_bronze.select([
    sum(
        when(
            col(c).isNull() | ((trim(col(c)))==""),1
            ).otherwise(0)
    ).alias(c)
    for c in df_bronze.columns
])
display(null_summary)

Employee_ID,First_Name,Last_Name,Email,Phone_Number,Department_Region,Birth_Year,Age,Join_Date,Salary,Performance_Score,Remote_Work,Status,batch_id,source_system,ingestion_timestamp
0,12,29,14,30,7,29,28,8,15,19,11,1,0,0,0


### Step 1B - Investigate Actual Values

### 1. Check duplicate Employee_IDs

In [0]:
from pyspark.sql.functions import col,count

duplicate_employee_ids = (
    df_bronze
    .groupBy("Employee_ID")
    .count()
    .filter(col("count")>1)
)

display(duplicate_employee_ids)

Employee_ID,count


### 2. Inspect name values

In [0]:
display(
    df_bronze.select(
        "First_Name",
        "Last_Name",
        "Email"
    )
)

First_Name,Last_Name,Email
Ganesh,Reddy,ganesh.reddy@example.com
Lakshmi,Nathan,lakshmi.nathan@example.com
Meena,Joshi,meena.joshi@example.com
Rahul,Bose,rahul.bose@example.com
Shalini,Verma,shalini.verma@example.com
Kiran,Rao,kiran.rao@example.com
Sanjay,Reddy,sanjay.reddy@example.com
Prakash,Sharma,prakash.sharma@example.com
Nisha,Iyer,nisha.iyer@example.com
Nisha,null,nisha@example.com


### 3. Inspect Phone_Number

In [0]:
display(
    df_bronze.select("Phone_Number").distinct()
)

Phone_Number
9542351161
8255341928
7884969653
6462704828
7278248963
8738299737
6801326773
7191361939
8251354278
6164005242


### 4. Inspect Department_Region

In [0]:
display(
    df_bronze.select("Department_Region").distinct()
)

Department_Region
Support-Karnataka
HR-Punjab
Sales-Delhi
Operations-Rajasthan
Support-Tamil Nadu
DevOps-Karnataka
HR-Tamil Nadu
Operations-Punjab
Marketing-Delhi
Admin-Karnataka


### 5. Inspect Birth_Year and Age

In [0]:
display(
    df_bronze.select(
        "Birth_Year",
        "Age"
    )
)

Birth_Year,Age
null,59
2004,22
1985,41
1989,37
1996,null
1969,57
1986,40
1996,30
1973,53
1973,53


### 6. Inspect Join_Date

In [0]:
display(
    df_bronze.select("Join_Date").distinct()
)

Join_Date
8/31/2020
2/26/2017
3/29/2025
10/26/2017
11/8/2024
7/31/2017
2/23/2017
11/11/2025
7/19/2018
4/29/2023


### 7. Inspect Salary

In [0]:
display(
    df_bronze.select("Salary").distinct()
)

Salary
100252.34
113071.48
142863.71
146384.8
148165.17
null
48776.11
32310.41
141127.29
94152.96


### 8. Inspect Performance_Score

In [0]:
display(
    df_bronze.select("Performance_Score").distinct()
)

Performance_Score
Poor
Average
Excelent
Good
Excellent
NA
5
null


### 9. Inspect Remote_Work

In [0]:
display(
    df_bronze.select("Remote_Work").distinct()
)

Remote_Work
Yes
No
1
N
Y
TRUE
FALSE
0
null


### 10. Inspect Status

In [0]:
display(
    df_bronze.select("Status").distinct()
)

Status
Inactive
Pending
On Leave
Active
pending
Onleave
active
Actve
INACTIVE
Terminated


### Step 2 - Employee_ID

Our rules are:

- Employee_ID must not be null/blank.
- Employee_ID must be unique.
- Failure of either condition → hard reject / quarantine.

### Step 2.1 - Check duplicate Employee_IDs

In [0]:
from pyspark.sql.functions import col,count

duplicate_ids = (
    df_bronze
    .groupBy("Employee_ID")
    .agg(count("*").alias("id_count"))
    .filter(col("id_count")>1)
)

display(duplicate_ids)

Employee_ID,id_count


### Step 2.2 - Create the Employee_ID validity flag

In [0]:
from pyspark.sql.functions import trim,when

df_silver = df_bronze.withColumn(
    "is_valid_employee_id",
    when(
        col("Employee_ID").isNotNull() &
        (trim(col("Employee_ID"))!=""),
        True
    ).otherwise(False)
)

### Step 3 - First_Name / Last_Name

### Step 3.1 - Investigate missing names

From Step 1, we already know:

- First_Name → 12 missing/blank
- Last_Name → 29 missing/blank

In [0]:
from pyspark.sql.functions import col,trim

missing_names = df_bronze.filter(
    (col("First_Name").isNull()) |
    (trim(col("First_Name"))=="") |
    (col("Last_name").isNull()) |
    (trim(col("Last_Name"))=="")
)

display(missing_names.select(
    "Employee_ID",
    "First_Name",
    "Last_Name",
    "Email"
))

Employee_ID,First_Name,Last_Name,Email
EMP3009,Nisha,null,nisha@example.com
EMP3038,Swathi,null,swathi.verma@example.com
EMP3059,Kiran,null,kiran@example.com
EMP3076,Kavya,null,kavya@example.com
EMP3186,Manoj,null,manoj@example.com
EMP3216,null,Mehta,naveen.mehta@example.com
EMP3233,null,Mehta,karthik.mehta@example.com
EMP3242,Arun,null,arun@example.com
EMP3249,Lakshmi,null,lakshmi@example.com
EMP3253,null,Pillai,lakshmi.pillai@example.com


### Step 3.2 - Trim existing names

In [0]:
from pyspark.sql.functions import trim

df_silver = (
    df_bronze
    .withColumn("First_Name",trim(col("First_Name")))
    .withColumn("Last_Name",trim(col("Last_Name")))
)

### Step 3.3 - Recover Missing Names from Email

In [0]:
from pyspark.sql.functions import split,col

display(
    df_silver.select(
        "Employee_ID",
        "First_Name",
        "Last_Name",
        "Email",
        split(col("Email"),"@")[0].alias("email_name_parts")
    )
)

Employee_ID,First_Name,Last_Name,Email,email_name_parts
EMP3000,Ganesh,Reddy,ganesh.reddy@example.com,ganesh.reddy
EMP3001,Lakshmi,Nathan,lakshmi.nathan@example.com,lakshmi.nathan
EMP3002,Meena,Joshi,meena.joshi@example.com,meena.joshi
EMP3003,Rahul,Bose,rahul.bose@example.com,rahul.bose
EMP3004,Shalini,Verma,shalini.verma@example.com,shalini.verma
EMP3005,Kiran,Rao,kiran.rao@example.com,kiran.rao
EMP3006,Sanjay,Reddy,sanjay.reddy@example.com,sanjay.reddy
EMP3007,Prakash,Sharma,prakash.sharma@example.com,prakash.sharma
EMP3008,Nisha,Iyer,nisha.iyer@example.com,nisha.iyer
EMP3009,Nisha,null,nisha@example.com,nisha


### Step 3.4 - Recover Missing Names from Email

In [0]:
from pyspark.sql.functions import split,col,trim,when

df_silver = (
    df_silver
    .withColumn(
        "email_username",
        split(col("Email"),"@").getItem(0)
    )
)

Now recover the missing First_Name:

In [0]:
df_silver = df_silver.withColumn(
    "First_Name",
    when(
        (
            col("First_Name").isNull() |
            (trim(col("First_Name"))=="")
        ) &
        col("email_username").contains('.'),
        split(col("email_username"),"\\.").getItem(0)
    ).otherwise(col("First_Name"))
)

In [0]:
df_silver = df_silver.withColumn(
    "Last_Name",
    when(
        (
        (col("Last_Name").isNull()) |
        (trim(col("Last_Name"))=="")
        ) &
        col("email_username").contains('.'),
        split(col("email_username"),'\\.').getItem(1)
    ).otherwise(col("Last_Name"))
)

In [0]:
display(
    df_silver.select(
        "Employee_ID",
        "First_Name",
        "Last_Name",
        "Email"
    )
)

Employee_ID,First_Name,Last_Name,Email
EMP3000,Ganesh,Reddy,ganesh.reddy@example.com
EMP3001,Lakshmi,Nathan,lakshmi.nathan@example.com
EMP3002,Meena,Joshi,meena.joshi@example.com
EMP3003,Rahul,Bose,rahul.bose@example.com
EMP3004,Shalini,Verma,shalini.verma@example.com
EMP3005,Kiran,Rao,kiran.rao@example.com
EMP3006,Sanjay,Reddy,sanjay.reddy@example.com
EMP3007,Prakash,Sharma,prakash.sharma@example.com
EMP3008,Nisha,Iyer,nisha.iyer@example.com
EMP3009,Nisha,null,nisha@example.com


In [0]:
# As we have used email_username as a temporary column, we can drop it now.
df_silver = df_silver.drop("email_username")
display(df_silver)

Employee_ID,First_Name,Last_Name,Email,Phone_Number,Department_Region,Birth_Year,Age,Join_Date,Salary,Performance_Score,Remote_Work,Status,batch_id,source_system,ingestion_timestamp
EMP3000,Ganesh,Reddy,ganesh.reddy@example.com,9542351161,Support-Karnataka,null,59,8/31/2020,100252.34,Poor,Yes,Inactive,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3001,Lakshmi,Nathan,lakshmi.nathan@example.com,8255341928,HR-Punjab,2004,22,2/26/2017,113071.48,Average,No,Pending,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3002,Meena,Joshi,meena.joshi@example.com,7884969653,Sales-Delhi,1985,41,3/29/2025,142863.71,Poor,1,Inactive,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3003,Rahul,Bose,rahul.bose@example.com,6462704828,Operations-Rajasthan,1989,37,10/26/2017,146384.8,Excelent,N,On Leave,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3004,Shalini,Verma,shalini.verma@example.com,7278248963,Support-Tamil Nadu,1996,null,11/8/2024,148165.17,Good,No,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3005,Kiran,Rao,kiran.rao@example.com,8738299737,DevOps-Karnataka,1969,57,7/31/2017,null,Excellent,Y,Inactive,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3006,Sanjay,Reddy,sanjay.reddy@example.com,6801326773,HR-Tamil Nadu,1986,40,2/23/2017,48776.11,Average,Yes,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3007,Prakash,Sharma,prakash.sharma@example.com,7191361939,Operations-Punjab,1996,30,11/11/2025,32310.41,Average,TRUE,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3008,Nisha,Iyer,nisha.iyer@example.com,8251354278,Marketing-Delhi,1973,53,7/19/2018,141127.29,null,No,Inactive,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3009,Nisha,null,nisha@example.com,6164005242,Admin-Karnataka,1973,53,4/29/2023,94152.96,Average,Y,On Leave,BATCH_001,employee_app,2026-09-26T12:44:46.516Z


### Step 3.5 - Flag Unrecoverable Names

In [0]:
from pyspark.sql.functions import col,trim,when

df_silver = df_silver.withColumn(
    "is_unrecoverable_name",
    when(
        (
        col("First_name").isNull() |
        (trim(col("First_Name")) == "")
        ) |
        (
        col("Last_Name").isNull() |
        (trim(col("Last_Name")) == "")
        ),
        True
    ).otherwise(False)
)

In [0]:
display(
    df_silver
    .filter(col("is_unrecoverable_name") == True)
    .select(
        "Employee_ID",
        "First_Name",
        "Last_Name",
        "Email",
        "is_unrecoverable_name"
    )
)

Employee_ID,First_Name,Last_Name,Email,is_unrecoverable_name
EMP3009,Nisha,null,nisha@example.com,true
EMP3059,Kiran,null,kiran@example.com,true
EMP3076,Kavya,null,kavya@example.com,true
EMP3186,Manoj,null,manoj@example.com,true
EMP3242,Arun,null,arun@example.com,true
EMP3249,Lakshmi,null,lakshmi@example.com,true
EMP3301,Naveen,null,naveen@example.com,true
EMP3383,Manoj,null,manoj@example.com,true
EMP3492,Uma,null,uma@example.com,true
EMP3549,Arjun,null,arjun@example.com,true


### Step 4 - Email

- Recover missing Email using First_Name and Last_Name when both are available.
- If Email is still missing because the required information is unavailable, flag it as unrecoverable.
- Validate the remaining emails using a regex pattern.

### Step 4.1 - Investigate missing Email

In [0]:
display(
    df_silver
    .filter(
        col("Email").isNull() |
        (trim(col("Email"))=="")
    )
    .select(
        "Employee_ID",
        "First_Name",
        "Last_Name",
        "Email"
    )
)

Employee_ID,First_Name,Last_Name,Email
EMP3012,Arjun,Reddy,null
EMP3070,Shalini,Sharma,null
EMP3084,Arun,Rao,null
EMP3111,Pooja,Mehta,null
EMP3193,Uma,Raman,null
EMP3314,Arun,Menon,null
EMP3365,Vikram,Patel,null
EMP3385,Priya,Chatterjee,null
EMP3437,Deepa,Nair,null
EMP3447,Pooja,Raman,null


### Step 4.2 - Recover Missing Email

In [0]:
from pyspark.sql.functions import col,trim,when,concat,lit,lower

df_silver = df_silver.withColumn(
    "Email",
    when(
        (
        col("Email").isNull() |
        (trim(col("Email")) =="")
        ) &
        col("First_Name").isNotNull() &
        col("Last_Name").isNotNull() &
        (trim(col("First_Name"))!="") &
        (trim(col("Last_Name"))!="") , 
        concat(
            lower(trim(col("First_Name"))),
            lit("."),
            lower(trim(col("Last_Name"))),
            lit("@example.com")
        )
    ).otherwise(col("Email"))
)

In [0]:
display(
    df_silver
    .select(
        "Employee_ID",
        "First_Name",
        "Last_Name",
        "Email"
    )
    .filter(
        col("Email").isNull()
    )
)

Employee_ID,First_Name,Last_Name,Email


### Step 4.3 - Flag Unrecoverable Email

In [0]:
from pyspark.sql.functions import col,trim,when

df_silver = df_silver.withColumn(
    "is_unrecoverable_email",
    when(
        col("Email").isNull() |
        (trim(col("Email"))==""),
        True
    ).otherwise(False)
)

In [0]:
display(
    df_silver
    .filter(col("is_unrecoverable_email") == True)
    .select(
        "Employee_ID",
        "First_Name",
        "Last_Name",
        "Email",
        "is_unrecoverable_email"
    )
)

Employee_ID,First_Name,Last_Name,Email,is_unrecoverable_email


### Step 4.4 - Validate Email Format

In [0]:
email_pattern = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
df_silver = df_silver.withColumn(
    "is_valid_email",
    when(
        col("Email").isNull() |
        (trim(col("Email"))==""),
        False
    )
    .when(
        trim(col("Email")).rlike(email_pattern),
        True
    ).otherwise(False)
)

In [0]:
display(
    df_silver
    .filter(col("is_valid_email") == False)
    .select(
        "Employee_ID",
        "First_Name",
        "Last_Name",
        "Email",
        "is_unrecoverable_email",
        "is_valid_email"
    )
)

Employee_ID,First_Name,Last_Name,Email,is_unrecoverable_email,is_valid_email


### Step 5  Phone_Number

- Remove the +91 prefix.
- Remove dashes.
- Check that the result has exactly 10 digits and starts with 6, 7, 8, or 9.

### Step 5.1 - Investigate the actual phone values

In [0]:
display(
    df_silver
    .select(
        "Employee_ID",
        "Phone_Number"
    )
)

Employee_ID,Phone_Number
EMP3000,9542351161
EMP3001,8255341928
EMP3002,7884969653
EMP3003,6462704828
EMP3004,7278248963
EMP3005,8738299737
EMP3006,6801326773
EMP3007,7191361939
EMP3008,8251354278
EMP3009,6164005242


In [0]:
display(
    df_silver
    .filter(
        col("Phone_Number").isNull() |
        (trim(col("Phone_Number")) == "")
    )
    .select(
        "Employee_ID",
        "Phone_Number"
    )
)

Employee_ID,Phone_Number
EMP3056,null
EMP3072,null
EMP3089,null
EMP3170,null
EMP3178,null
EMP3234,null
EMP3269,null
EMP3276,null
EMP3311,null
EMP3381,null


### Step 5.2 - Remove +91 and Dashes

In [0]:
from pyspark.sql.functions import col,regexp_replace

df_silver = df_silver.withColumn(
    "Phone_Number",
    regexp_replace(col("Phone_Number"),r"^\+91","")
)

df_silver = df_silver.withColumn(
    "Phone_Number",
    regexp_replace(col("Phone_Number"),'-','')
)

In [0]:
display(
    df_silver
    .select(
        "Employee_ID",
        "Phone_Number"
    )
)

Employee_ID,Phone_Number
EMP3000,9542351161
EMP3001,8255341928
EMP3002,7884969653
EMP3003,6462704828
EMP3004,7278248963
EMP3005,8738299737
EMP3006,6801326773
EMP3007,7191361939
EMP3008,8251354278
EMP3009,6164005242


### Step 5.3 - Check Phone Number Length

In [0]:
from pyspark.sql.functions import length,col,when

df_silver = df_silver.withColumn(
    "is_valid_phone_length",
    when(
        col("Phone_Number").isNull() |
        (length(col("Phone_Number")) !=10),
        False
    ).otherwise(True)
)

### Invalid Phone length

In [0]:
display(
    df_silver
    .select(
        "Employee_ID",
        "Phone_Number",
        "is_valid_phone_length"
    )
    .filter(col("is_valid_phone_length")==False)
)

Employee_ID,Phone_Number,is_valid_phone_length
EMP3056,null,false
EMP3062,68707286797,false
EMP3072,null,false
EMP3089,null,false
EMP3101,885646799,false
EMP3170,null,false
EMP3178,null,false
EMP3234,null,false
EMP3269,null,false
EMP3276,null,false


### Step 5.4 - Check the First Digit

In [0]:
from pyspark.sql.functions import substring,col

display(
    df_silver
    .select(
        "Employee_ID",
        "Phone_Number",
        substring(col("Phone_Number"),1,1).alias("first_digit")
    )
)

Employee_ID,Phone_Number,first_digit
EMP3000,9542351161,9
EMP3001,8255341928,8
EMP3002,7884969653,7
EMP3003,6462704828,6
EMP3004,7278248963,7
EMP3005,8738299737,8
EMP3006,6801326773,6
EMP3007,7191361939,7
EMP3008,8251354278,8
EMP3009,6164005242,6


### Step 5.5 - Create the final is_valid_phone flag

In [0]:
from pyspark.sql.functions import col,length,substring,when

df_silver = df_silver.withColumn(
    "is_valid_phone",
    when(
        col("Phone_Number").isNull() |
        (length(col("Phone_Number"))!=10) |
        (~substring(col("Phone_Number"),1,1).isin("6","7","8","9")),
        False
    ).otherwise(True)
)

### Check invalid phone numbers

In [0]:
display(
    df_silver
    .filter(col("is_valid_phone")==False)
    .select(
        "Employee_ID",
        "Phone_Number",
        "is_valid_phone"
    )
)

Employee_ID,Phone_Number,is_valid_phone
EMP3041,0154665905,false
EMP3056,null,false
EMP3062,68707286797,false
EMP3072,null,false
EMP3085,0103761379,false
EMP3089,null,false
EMP3097,0186179327,false
EMP3101,885646799,false
EMP3113,0490028772,false
EMP3139,0434132611,false


### Step 6 - Department_Region

### Step 6.1 - Investigate the actual values

In [0]:
display(
    df_silver
    .select(
        "Employee_ID","Department_Region"
    )
)

Employee_ID,Department_Region
EMP3000,Support-Karnataka
EMP3001,HR-Punjab
EMP3002,Sales-Delhi
EMP3003,Operations-Rajasthan
EMP3004,Support-Tamil Nadu
EMP3005,DevOps-Karnataka
EMP3006,HR-Tamil Nadu
EMP3007,Operations-Punjab
EMP3008,Marketing-Delhi
EMP3009,Admin-Karnataka


In [0]:
display(
    df_silver
    .filter(
        col("Department_Region").isNull() |
        (trim(col("Department_Region"))=="")
    )
    .select(
        "Employee_ID",
        "Department_Region"
    )
)

Employee_ID,Department_Region
EMP3341,null
EMP3455,null
EMP3504,null
EMP3621,null
EMP3655,null
EMP3755,null
EMP3999,null


### Step 6.2 - Split into Department and Region

In [0]:
from pyspark.sql.functions import split,col,trim

df_silver = (
    df_silver
    .withColumn(
        "Department",
        trim(split(col("Department_Region"),'-').getItem(0))
    )
    .withColumn(
        "Region",
        trim(split(col("Department_Region"),'-').getItem(1))
    )
)

### Step 6.3 - Flag missing Department_Region

In [0]:
from pyspark.sql.functions import when

df_silver = df_silver.withColumn(
    "is_valid_department_region",
    when(
        col("Department_Region").isNull() |
        trim(col("Department_Region")==""),
        True
    ).otherwise(False)
)

In [0]:
display(
    df_silver
    .filter(col("is_valid_department_region") == True)
    .select(
        "Employee_ID",
        "Department_Region",
        "Department",
        "Region",
        "is_valid_department_region"
    )
)

Employee_ID,Department_Region,Department,Region,is_valid_department_region
EMP3341,null,null,null,true
EMP3455,null,null,null,true
EMP3504,null,null,null,true
EMP3621,null,null,null,true
EMP3655,null,null,null,true
EMP3755,null,null,null,true
EMP3999,null,null,null,true


### Step 7 - Birth_Year / Age

Our cleaning rule is:

- If Age is missing but Birth_Year exists → calculate Age.
- If Birth_Year is missing but Age exists → calculate Birth_Year.
- If both are missing → flag as unrecoverable.
- If an existing Age is invalid - negative, 0, over ~100, or non-numeric - flag it.
- We will not immediately remove the row.

### Step 7.1 - Investigate the actual values

In [0]:
display(
    df_silver
    .filter(
        col("Birth_Year").isNull() |
        trim(col("Birth_Year")=="") |
        col("Age").isNull() |
        (trim(col("Age"))=="")
    )
    .select(
        "Employee_ID",
        "Birth_Year",
        "Age"
    )
)

Employee_ID,Birth_Year,Age
EMP3000,null,59
EMP3004,1996,null
EMP3027,1994,null
EMP3031,null,31
EMP3032,null,44
EMP3033,2003,null
EMP3034,null,null
EMP3047,null,32
EMP3086,1969,null
EMP3088,1997,null


In [0]:
display(
    df_silver
    .select("Birth_Year","Age")
    .distinct()
)

Birth_Year,Age
null,59
2004,22
1985,41
1989,37
1996,null
1969,57
1986,40
1996,30
1973,53
1974,52


### Step 7.2 - Recover Missing Age from Birth_Year

In [0]:
from pyspark.sql.functions import col,trim,when,year,current_date


df_silver = df_silver.withColumn(
    "Age",
    when(
    (
        col("Age").isNull() |
        (trim(col("Age"))=="") |
        (trim(col("Age"))=='N/A')
    )
    & 
    col("Birth_Year").isNotNull() &
    (trim(col("Birth_Year"))!="") &
    (trim(col("Birth_Year"))!='N/A'),
    year(current_date())-col("Birth_Year").cast("int")
    ).otherwise(col("Age"))
)

In [0]:
display(
    df_silver
    .select(
        "Employee_ID",
        "Birth_Year",
        "Age"
    )
)

Employee_ID,Birth_Year,Age
EMP3000,null,59
EMP3001,2004,22
EMP3002,1985,41
EMP3003,1989,37
EMP3004,1996,30
EMP3005,1969,57
EMP3006,1986,40
EMP3007,1996,30
EMP3008,1973,53
EMP3009,1973,53


### Step 7.3 - Recover Missing Birth_Year from Age

In [0]:
from pyspark.sql.functions import col,trim,when,year,current_date

df_silver = df_silver.withColumn(
    "Birth_Year",
    when(
        (
    col("Birth_Year").isNull() |
    (trim(col("Birth_Year"))=="") |
    (trim(col("Birth_Year"))=="N/A")
        ) &
        col("Age").isNotNull() &
        (trim(col("Age"))!="") &
        (trim(col("Age"))!='N/A'),
        year(current_date())-col("Age").cast("int")
    ).otherwise(col("Birth_Year"))
)

In [0]:
display(
    df_silver
    .select(
        "Employee_ID",
        "Birth_Year",
        "Age"
    )
)

Employee_ID,Birth_Year,Age
EMP3000,1967,59
EMP3001,2004,22
EMP3002,1985,41
EMP3003,1989,37
EMP3004,1996,30
EMP3005,1969,57
EMP3006,1986,40
EMP3007,1996,30
EMP3008,1973,53
EMP3009,1973,53


### Step 7.4 - Flag Unrecoverable Age / Birth_Year

In [0]:
from pyspark.sql.functions import col,trim,when

df_silver = df_silver.withColumn(
    "is_unrecoverable_age",
    when(
        (
        col("Age").isNull() |
        (trim(col("Age"))=="") |
        (trim(col("Age")) == "N/A")
        ) &
        col("Birth_Year").isNull() |
        (trim(col("Birth_Year"))=="") |
        (trim(col("Birth_Year"))=='N/A'),
        True
    ).otherwise(False)
)

In [0]:
display(
    df_silver
    .filter(col("is_unrecoverable_age") == True)
    .select(
        "Employee_ID",
        "Birth_Year",
        "Age",
        "is_unrecoverable_age"
    )
)

Employee_ID,Birth_Year,Age,is_unrecoverable_age
EMP3034,null,null,true
EMP3094,null,null,true
EMP3353,null,null,true
EMP3442,null,null,true
EMP3480,null,null,true
EMP3765,null,null,true


### Step 7.5 - Validate Age

- Age must be numeric.
- Age must be greater than 0.
- Age must not be greater than 100.
- "N/A" is invalid.

In [0]:
from pyspark.sql.functions import col,expr,when

df_silver = df_silver.withColumn(
    "is_valid_age",
    when(
        expr("try_cast(Age as INT)").isNull(),
        False
    )
    .when(
        (expr("try_cast(Age as INT)") <=0) |
        (expr("try_cast(Age as INT)") > 100),
        False
    ).otherwise(True)
)

In [0]:
display(
    df_silver
    .filter(
        col("is_valid_age") == False
    )
    .select(
        "Employee_ID",
        "Birth_Year",
        "Age",
        "is_unrecoverable_age",
        "is_valid_age"
    )
)

Employee_ID,Birth_Year,Age,is_unrecoverable_age,is_valid_age
EMP3034,null,null,true,false
EMP3094,null,null,true,false
EMP3115,1971,0,false,false
EMP3161,1978,0,false,false
EMP3283,1979,0,false,false
EMP3353,null,null,true,false
EMP3426,1972,-5,false,false
EMP3442,null,null,true,false
EMP3453,1990,150,false,false
EMP3480,null,null,true,false


### Step 8 - Join_Date

Our cleaning plan has two rules:

- Parse the date using the source format M/d/yyyy.
- If the date is missing or is after today, flag it as invalid.

### Step 8.1 - Inspect the actual Join_Date values

In [0]:
from pyspark.sql.functions import col,trim

display(
    df_silver
    .select(
        "Employee_ID",
        "Join_Date"
    )
)

Employee_ID,Join_Date
EMP3000,8/31/2020
EMP3001,2/26/2017
EMP3002,3/29/2025
EMP3003,10/26/2017
EMP3004,11/8/2024
EMP3005,7/31/2017
EMP3006,2/23/2017
EMP3007,11/11/2025
EMP3008,7/19/2018
EMP3009,4/29/2023


In [0]:
display(
    df_silver
    .filter(
        col("Join_Date").isNull() |
        (trim(col("Join_Date"))=="")
    )
    .select(
        "Employee_ID",
        "Join_Date"
    )
)

Employee_ID,Join_Date
EMP3038,null
EMP3082,null
EMP3083,null
EMP3246,null
EMP3280,null
EMP3367,null
EMP3690,null
EMP3850,null


### Step 8.2 - Parse Join_Date

In [0]:
from pyspark.sql.functions import col,to_date

df_silver = df_silver.withColumn(
    "Join_Date",
    to_date(trim(col("Join_Date")),'M/d/yyyy')
)

In [0]:
display(
    df_silver
    .select(
        "Employee_ID",
        "Join_Date"
    )
)

Employee_ID,Join_Date
EMP3000,2020-08-31
EMP3001,2017-02-26
EMP3002,2025-03-29
EMP3003,2017-10-26
EMP3004,2024-11-08
EMP3005,2017-07-31
EMP3006,2017-02-23
EMP3007,2025-11-11
EMP3008,2018-07-19
EMP3009,2023-04-29


### Step 8.3 - Validate missing and future dates

In [0]:
from pyspark.sql.functions import col,current_date,when

df_silver = df_silver.withColumn(
    "is_valid_join_date",
    when(
        (
        col('Join_Date').isNull() |
        (col('Join_Date') > current_date())
        ),
        False
    ).otherwise(True)
)

In [0]:
display(
    df_silver
    .filter(col("is_valid_join_date") == False)
    .select(
        "Employee_ID",
        "Join_Date",
        "is_valid_join_date"
    )
)

Employee_ID,Join_Date,is_valid_join_date
EMP3038,null,false
EMP3069,2029-05-31,false
EMP3082,null,false
EMP3083,null,false
EMP3196,2029-11-27,false
EMP3205,2029-11-23,false
EMP3219,2029-03-19,false
EMP3231,2028-02-26,false
EMP3246,null,false
EMP3280,null,false


### Step 9 - Salary

Our rules are:

- If Salary is blank → flag it.
- If Salary is negative → flag it.
- If Salary is non-numeric, such as "N/A" → flag it.
- We do not drop the row yet.

### Step 9.1 - Inspect the actual Salary values

In [0]:
display(
    df_silver
    .select(
        "Employee_ID",
        "Salary"
    )
)

Employee_ID,Salary
EMP3000,100252.34
EMP3001,113071.48
EMP3002,142863.71
EMP3003,146384.8
EMP3004,148165.17
EMP3005,null
EMP3006,48776.11
EMP3007,32310.41
EMP3008,141127.29
EMP3009,94152.96


In [0]:
display(
    df_silver
    .filter(
        col("Salary").isNull() |
        (trim(col("Salary")) == "")
    )
    .select(
        "Employee_ID",
        "Salary"
    )
)

Employee_ID,Salary
EMP3005,null
EMP3182,null
EMP3322,null
EMP3364,null
EMP3390,null
EMP3407,null
EMP3431,null
EMP3522,null
EMP3547,null
EMP3633,null


### Step 9.2 - Check Whether Salary Is Numeric

Now we need to distinguish between:

- numeric salary values
- non-numeric values such as "N/A"
- blank/null values

Since Salary is a string, we'll use try_cast().

In [0]:
from pyspark.sql.functions import col,expr

display(
    df_silver
    .select(
        "Employee_ID",
        "Salary",
        expr("try_cast(Salary AS double)").alias("salary_numeric")
    )
)

Employee_ID,Salary,salary_numeric
EMP3000,100252.34,100252.34
EMP3001,113071.48,113071.48
EMP3002,142863.71,142863.71
EMP3003,146384.8,146384.8
EMP3004,148165.17,148165.17
EMP3005,null,null
EMP3006,48776.11,48776.11
EMP3007,32310.41,32310.41
EMP3008,141127.29,141127.29
EMP3009,94152.96,94152.96


### Check only the non-numeric values

In [0]:
display(
    df_silver
    .filter(
        col("Salary").isNotNull() &
        (trim(col("Salary"))!="") &
        expr("try_cast(Salary as double)").isNull()
    )
    .select(
        "Employee_ID",
        "Salary"
    )
)

Employee_ID,Salary
EMP3229,N/A
EMP3543,N/A
EMP3866,N/A
EMP3932,N/A


In [0]:
from pyspark.sql.functions import col,trim,when,expr

df_silver = df_silver.withColumn(
    "is_valid_salary",
    when(
        (
        col("Salary").isNull() |
        (trim(col("Salary")) == "")
        ),
        False
    )
    .when(
        expr("try_cast(Salary as double)").isNull(),
        False
    )
    .when(
        expr("try_cast(Salary as double)") < 0,
        False
    ).otherwise(True)
)

### Check invalid salaries

In [0]:
display(
    df_silver.filter(col("is_valid_salary") == False)
    .select(
        "Employee_ID",
        "Salary",
        "is_valid_salary"
    )
)

Employee_ID,Salary,is_valid_salary
EMP3005,null,false
EMP3133,-50038.8,false
EMP3182,null,false
EMP3229,N/A,false
EMP3322,null,false
EMP3345,-89044.45,false
EMP3364,null,false
EMP3390,null,false
EMP3407,null,false
EMP3431,null,false


### Step 10 - Performance_Score

### Step 10.1 - Inspect the actual values

In [0]:
display(
    df_silver
    .select(
        "Employee_ID",
        "Performance_Score"
    )
)

Employee_ID,Performance_Score
EMP3000,Poor
EMP3001,Average
EMP3002,Poor
EMP3003,Excelent
EMP3004,Good
EMP3005,Excellent
EMP3006,Average
EMP3007,Average
EMP3008,null
EMP3009,Average


In [0]:
display(
    df_silver
    .select("Performance_Score")
    .distinct()
)

Performance_Score
Poor
Average
Excelent
Good
Excellent
NA
5
null


### Step 10.2 - Fix the known typo

In [0]:
from pyspark.sql.functions import col,trim,when

df_silver = df_silver.withColumn(
    "Performance_Score",
    when(
        trim(col("Performance_Score")) == 'Excelent',
        "Excellent"
    )
    .otherwise(trim(col("Performance_Score")))
)

In [0]:
display(
    df_silver
    .select(
        "Performance_Score"
    )
    .distinct()
)

Performance_Score
Poor
Average
Excellent
Good
null
NA
5


### Step 10.3 - Validate Performance_Score

In [0]:
performance_score_list = ['Excellent','Good','Average','Poor']
df_silver = df_silver.withColumn(
    "is_valid_performance_score",
    when(
        col("Performance_Score").isin(performance_score_list),
        True
    ).otherwise(False)
)

In [0]:
display(
    df_silver
    .filter(col("is_valid_performance_score") == False)
    .select(
        "Employee_ID",
        "Performance_Score",
        "is_valid_performance_score"
    )
)

Employee_ID,Performance_Score,is_valid_performance_score
EMP3008,null,false
EMP3095,null,false
EMP3097,null,false
EMP3186,null,false
EMP3247,null,false
EMP3270,null,false
EMP3290,null,false
EMP3319,null,false
EMP3352,NA,false
EMP3407,null,false


### Step 11 - Remote_Work

Our rule is:

| Source value               | Standardized value |
| -------------------------- | ------------------ |
| `Yes` / `TRUE` / `Y` / `1` | `True`             |
| `No` / `FALSE` / `N` / `0` | `False`            |
| Blank / null               | `Unknown`          |


### Step 11.1 - Inspect the actual values

In [0]:
display(
    df_silver
    .select("Remote_Work")
    .distinct()
)

Remote_Work
Yes
No
1
N
Y
TRUE
FALSE
0
null


### Step 11.2 - Standardize the values

In [0]:
from pyspark.sql.functions import col,trim,upper,when

df_silver = df_silver.withColumn(
    "Remote_Work",
    trim(col("Remote_Work"))
)

df_silver = df_silver.withColumn(
    "Remote_Work",
    when(
        upper(col("Remote_Work")).isin("Yes","Y","1","TRUE"),
        "True"
    )
    .when(
        upper(col("Remote_Work")).isin("No","N","0","FALSE"),
        "False"
    )
    .when(
        col("Remote_Work").isNull() |
        (col("Remote_Work")==""),
        "Unknown"
    ).otherwise("Unknown")
)

In [0]:
display(
    df_silver
    .select(
        "Employee_ID",
        "Remote_Work"
    )
)

Employee_ID,Remote_Work
EMP3000,Unknown
EMP3001,Unknown
EMP3002,True
EMP3003,False
EMP3004,Unknown
EMP3005,True
EMP3006,Unknown
EMP3007,True
EMP3008,Unknown
EMP3009,True


In [0]:
display(
    df_silver
    .select("Remote_Work")
    .distinct()
)

Remote_Work
Unknown
True
False


In [0]:
display(
    df_silver
    .groupBy("Remote_Work")
    .count()
)

Remote_Work,count
Unknown,256
True,377
False,367


### Step 12 - Status

Our rules are:

- Remove leading/trailing spaces.
- Standardize the casing.

- Fix the known typo: Actve → Active
- Valid values are only:Active, Inactive, On Leave, Pending
- Anything else, including blank, should be flagged as invalid.
- We do not quarantine the row yet. That decision happens at the end.

### Step 12.1 - Inspect the actual values

In [0]:
display(
    df_silver
    .select("Status")
    .distinct()
)

Status
Inactive
Pending
On Leave
Active
pending
Onleave
active
Actve
INACTIVE
Terminated


### Step 12.2 - Corrected Standardization

In [0]:
from pyspark.sql.functions import col,trim,initcap,when

df_silver = df_silver.withColumn(
    "Status",
    initcap(trim(col("Status")))
)

# After this run below cell

df_silver = df_silver.withColumn(
    "Status",
    when(
        col("Status") == "Actve",
        "Active"
    )
    .when(
        col("Status") == "Onleave",
        "On Leave"
    )
    .otherwise(col("Status"))
)

In [0]:
display(
    df_silver
    .select("Status")
    .distinct()
)

Status
Inactive
Pending
On Leave
Active
null
Terminated
N/a


### Step 12.3 - Create the validation flag

In [0]:
from pyspark.sql.functions import when,col

status_list = ['Active','Inactive','On Leave','Pending']

df_silver = df_silver.withColumn(
    "is_valid_status",
    when(
        col("Status").isin(status_list),
        True
    )
    .otherwise(False)
)

In [0]:
display(
    df_silver
    .filter(col("is_valid_status") == False)
    .select(
        "Employee_ID",
        "Status",
        "is_valid_status"
    )
)

Employee_ID,Status,is_valid_status
EMP3469,null,false
EMP3694,Terminated,false
EMP3709,N/a,false
EMP3768,Terminated,false
EMP3823,Terminated,false


### Step 13 - Create rejection reason

### Step 13.1 - Create rejection_reason

In [0]:
from pyspark.sql.functions import concat_ws,when,col

# concat_ws which is used to concatenate multiple input strings into single string column

df_silver = df_silver.withColumn(
    "rejection_reason",
    concat_ws(
        '; ',
        when(col("is_unrecoverable_name") == True,"Unrecoverable Name"),
        when(col("is_unrecoverable_email") == True,"Unrecoverable Email"),
        when(col("is_unrecoverable_age") == True ,"Unrecoverable Age"),
        when(col("is_valid_email") == False ,"Invalid Email"),
        when(col("is_valid_phone") == False, "Invalid Phone"),
        when(col("is_valid_department_region") == True,"Invalid Department/Region"),
        when(col("is_valid_age") == False,"Invalid Age"),
        when(col("is_valid_join_date") == False,"Invalid Join Date"),
        when(col("is_valid_salary") == False, "Invalid Salary"),
        when(col("is_valid_performance_score") == False , "Invalid Performance Score"),
        when(col("is_valid_status") == False,"Invalid Status")
    )
)

In [0]:
display(df_silver)

Employee_ID,First_Name,Last_Name,Email,Phone_Number,Department_Region,Birth_Year,Age,Join_Date,Salary,Performance_Score,Remote_Work,Status,batch_id,source_system,ingestion_timestamp,is_unrecoverable_name,is_unrecoverable_email,is_valid_email,is_valid_phone_length,is_valid_phone,Department,Region,is_valid_department_region,is_unrecoverable_age,is_valid_age,is_valid_join_date,is_valid_salary,is_valid_performance_score,is_valid_status,rejection_reason
EMP3000,Ganesh,Reddy,ganesh.reddy@example.com,9542351161,Support-Karnataka,1967,59,2020-08-31,100252.34,Poor,Unknown,Inactive,BATCH_001,employee_app,2026-09-26T12:44:46.516Z,false,false,true,true,true,Support,Karnataka,false,false,true,true,true,true,true,
EMP3001,Lakshmi,Nathan,lakshmi.nathan@example.com,8255341928,HR-Punjab,2004,22,2017-02-26,113071.48,Average,Unknown,Pending,BATCH_001,employee_app,2026-09-26T12:44:46.516Z,false,false,true,true,true,HR,Punjab,false,false,true,true,true,true,true,
EMP3002,Meena,Joshi,meena.joshi@example.com,7884969653,Sales-Delhi,1985,41,2025-03-29,142863.71,Poor,True,Inactive,BATCH_001,employee_app,2026-09-26T12:44:46.516Z,false,false,true,true,true,Sales,Delhi,false,false,true,true,true,true,true,
EMP3003,Rahul,Bose,rahul.bose@example.com,6462704828,Operations-Rajasthan,1989,37,2017-10-26,146384.8,Excellent,False,On Leave,BATCH_001,employee_app,2026-09-26T12:44:46.516Z,false,false,true,true,true,Operations,Rajasthan,false,false,true,true,true,true,true,
EMP3004,Shalini,Verma,shalini.verma@example.com,7278248963,Support-Tamil Nadu,1996,30,2024-11-08,148165.17,Good,Unknown,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z,false,false,true,true,true,Support,Tamil Nadu,false,false,true,true,true,true,true,
EMP3005,Kiran,Rao,kiran.rao@example.com,8738299737,DevOps-Karnataka,1969,57,2017-07-31,null,Excellent,True,Inactive,BATCH_001,employee_app,2026-09-26T12:44:46.516Z,false,false,true,true,true,DevOps,Karnataka,false,false,true,true,false,true,true,Invalid Salary
EMP3006,Sanjay,Reddy,sanjay.reddy@example.com,6801326773,HR-Tamil Nadu,1986,40,2017-02-23,48776.11,Average,Unknown,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z,false,false,true,true,true,HR,Tamil Nadu,false,false,true,true,true,true,true,
EMP3007,Prakash,Sharma,prakash.sharma@example.com,7191361939,Operations-Punjab,1996,30,2025-11-11,32310.41,Average,True,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z,false,false,true,true,true,Operations,Punjab,false,false,true,true,true,true,true,
EMP3008,Nisha,Iyer,nisha.iyer@example.com,8251354278,Marketing-Delhi,1973,53,2018-07-19,141127.29,null,Unknown,Inactive,BATCH_001,employee_app,2026-09-26T12:44:46.516Z,false,false,true,true,true,Marketing,Delhi,false,false,true,true,true,false,true,Invalid Performance Score
EMP3009,Nisha,null,nisha@example.com,6164005242,Admin-Karnataka,1973,53,2023-04-29,94152.96,Average,True,On Leave,BATCH_001,employee_app,2026-09-26T12:44:46.516Z,true,false,true,true,true,Admin,Karnataka,false,false,true,true,true,true,true,Unrecoverable Name


### Step 13.3 - Separate Silver and Quarantine Rows

In [0]:
from pyspark.sql.functions import col

df_silver_final = df_silver.filter(
    col("rejection_reason") == ""
).select(
    "Employee_ID",
    "First_Name",
    "Last_Name",
    "Email",
    "Phone_Number",
    "Department",
    "Region",
    "Birth_Year",
    "Age",
    "Join_Date",
    "Salary",
    "Performance_Score",
    "Remote_Work",
    "Status",
    "batch_id",
    "source_system",
    "ingestion_timestamp"
)

In [0]:
display(df_silver_final)

Employee_ID,First_Name,Last_Name,Email,Phone_Number,Department,Region,Birth_Year,Age,Join_Date,Salary,Performance_Score,Remote_Work,Status,batch_id,source_system,ingestion_timestamp
EMP3000,Ganesh,Reddy,ganesh.reddy@example.com,9542351161,Support,Karnataka,1967,59,2020-08-31,100252.34,Poor,Unknown,Inactive,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3001,Lakshmi,Nathan,lakshmi.nathan@example.com,8255341928,HR,Punjab,2004,22,2017-02-26,113071.48,Average,Unknown,Pending,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3002,Meena,Joshi,meena.joshi@example.com,7884969653,Sales,Delhi,1985,41,2025-03-29,142863.71,Poor,True,Inactive,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3003,Rahul,Bose,rahul.bose@example.com,6462704828,Operations,Rajasthan,1989,37,2017-10-26,146384.8,Excellent,False,On Leave,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3004,Shalini,Verma,shalini.verma@example.com,7278248963,Support,Tamil Nadu,1996,30,2024-11-08,148165.17,Good,Unknown,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3006,Sanjay,Reddy,sanjay.reddy@example.com,6801326773,HR,Tamil Nadu,1986,40,2017-02-23,48776.11,Average,Unknown,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3007,Prakash,Sharma,prakash.sharma@example.com,7191361939,Operations,Punjab,1996,30,2025-11-11,32310.41,Average,True,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3010,Pooja,Patel,pooja.patel@example.com,7322602563,Sales,Kerala,1974,52,2019-02-02,137372.69,Average,Unknown,Pending,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3011,Hema,Reddy,hema.reddy@example.com,8294019655,Marketing,Telangana,1979,47,2019-06-21,66032.74,Average,True,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3012,Arjun,Reddy,arjun.reddy@example.com,7366299468,Support,Delhi,1969,57,2021-12-27,64239.46,Average,False,On Leave,BATCH_001,employee_app,2026-09-26T12:44:46.516Z


In [0]:
from pyspark.sql.functions import col

df_quarantine = df_silver.filter(
    col("rejection_reason") != ""
).select(
    "Employee_ID",
    "First_Name",
    "Last_Name",
    "Email",
    "Phone_Number",
    "Department_Region",
    "Department",
    "Region",
    "Birth_Year",
    "Age",
    "Join_Date",
    "Salary",
    "Performance_Score",
    "Remote_Work",
    "Status",
    "rejection_reason",
    "batch_id",
    "source_system",
    "ingestion_timestamp"
)

### Confirm the row counts

In [0]:
bronze_count = df_bronze.count()
silver_count = df_silver_final.count()
quarantine_count = df_quarantine.count()

print("Bronze rows          :",bronze_count)
print("Silver rows          :",silver_count)
print("Quarantine rows      :",quarantine_count)
print("Total output rows    :",silver_count+quarantine_count)

Bronze rows          : 1000
Silver rows          : 823
Quarantine rows      : 177
Total output rows    : 1000


### Step 14 - Write the Final Silver and Quarantine Tables

### 14.1 Write the Silver table

In [0]:
df_silver_final.write.format("delta").mode("overwrite").saveAsTable("workspace.data_analysis.silver_employees")

### 14.2 Write the Quarantine table

In [0]:
df_quarantine.write.format("delta").mode("overwrite").saveAsTable("workspace.data_analysis.employee_quarantine")

### 14.3 Confirm the table row counts

In [0]:
silver_table_count = spark.table(
    "workspace.data_analysis.silver_employees"
).count()

quarantine_table_count = spark.table(
    "workspace.data_analysis.employee_quarantine"
).count()


print("Silver table rows        :",silver_table_count)
print("Quarantine table rows    :",quarantine_table_count)
print("Total output rows        :",silver_table_count+quarantine_table_count)

Silver table rows        : 823
Quarantine table rows    : 177
Total output rows        : 1000


In [0]:
bronze_table_count = spark.table(
    "workspace.data_analysis.bronze_employees"
).count()
print("Bronze table rows       :",bronze_table_count)

Bronze table rows       : 1000


In [0]:
if bronze_table_count == silver_table_count + quarantine_table_count:
    print("Row count check passed")
else:
    print("Rows count check failed")

Row count check passed


In [0]:
%sql
select * from workspace.data_analysis.silver_employees;

Employee_ID,First_Name,Last_Name,Email,Phone_Number,Department,Region,Birth_Year,Age,Join_Date,Salary,Performance_Score,Remote_Work,Status,batch_id,source_system,ingestion_timestamp
EMP3000,Ganesh,Reddy,ganesh.reddy@example.com,9542351161,Support,Karnataka,1967,59,2020-08-31,100252.34,Poor,Unknown,Inactive,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3001,Lakshmi,Nathan,lakshmi.nathan@example.com,8255341928,HR,Punjab,2004,22,2017-02-26,113071.48,Average,Unknown,Pending,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3002,Meena,Joshi,meena.joshi@example.com,7884969653,Sales,Delhi,1985,41,2025-03-29,142863.71,Poor,True,Inactive,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3003,Rahul,Bose,rahul.bose@example.com,6462704828,Operations,Rajasthan,1989,37,2017-10-26,146384.8,Excellent,False,On Leave,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3004,Shalini,Verma,shalini.verma@example.com,7278248963,Support,Tamil Nadu,1996,30,2024-11-08,148165.17,Good,Unknown,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3006,Sanjay,Reddy,sanjay.reddy@example.com,6801326773,HR,Tamil Nadu,1986,40,2017-02-23,48776.11,Average,Unknown,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3007,Prakash,Sharma,prakash.sharma@example.com,7191361939,Operations,Punjab,1996,30,2025-11-11,32310.41,Average,True,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3010,Pooja,Patel,pooja.patel@example.com,7322602563,Sales,Kerala,1974,52,2019-02-02,137372.69,Average,Unknown,Pending,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3011,Hema,Reddy,hema.reddy@example.com,8294019655,Marketing,Telangana,1979,47,2019-06-21,66032.74,Average,True,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3012,Arjun,Reddy,arjun.reddy@example.com,7366299468,Support,Delhi,1969,57,2021-12-27,64239.46,Average,False,On Leave,BATCH_001,employee_app,2026-09-26T12:44:46.516Z


In [0]:
%sql
select * from workspace.data_analysis.employee_quarantine;

Employee_ID,First_Name,Last_Name,Email,Phone_Number,Department_Region,Department,Region,Birth_Year,Age,Join_Date,Salary,Performance_Score,Remote_Work,Status,rejection_reason,batch_id,source_system,ingestion_timestamp
EMP3005,Kiran,Rao,kiran.rao@example.com,8738299737,DevOps-Karnataka,DevOps,Karnataka,1969,57,2017-07-31,null,Excellent,True,Inactive,Invalid Salary,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3008,Nisha,Iyer,nisha.iyer@example.com,8251354278,Marketing-Delhi,Marketing,Delhi,1973,53,2018-07-19,141127.29,null,Unknown,Inactive,Invalid Performance Score,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3009,Nisha,null,nisha@example.com,6164005242,Admin-Karnataka,Admin,Karnataka,1973,53,2023-04-29,94152.96,Average,True,On Leave,Unrecoverable Name,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3034,Kiran,Raman,kiran.raman@example.com,8698478961,Support-Telangana,Support,Telangana,null,null,2019-07-04,78284.2,Poor,True,On Leave,Unrecoverable Age; Invalid Age,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3038,Swathi,verma,swathi.verma@example.com,9071596966,IT-Maharashtra,IT,Maharashtra,1983,43,null,118376.77,Good,False,Active,Invalid Join Date,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3041,Manoj,Subramaniam,manoj.subramaniam@example.com,0154665905,IT-Kerala,IT,Kerala,1997,29,2016-02-28,126370.85,Excellent,False,Pending,Invalid Phone,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3056,Anjali,Verma,anjali.verma@example.com,null,DevOps-Tamil Nadu,DevOps,Tamil Nadu,1972,54,2023-12-08,137345.89,Poor,True,Pending,Invalid Phone,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3059,Kiran,null,kiran@example.com,9206724004,Finance-West Bengal,Finance,West Bengal,1995,31,2025-10-23,60905.63,Excellent,Unknown,Inactive,Unrecoverable Name,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3062,Hema,Rao,hema.rao@example.com,68707286797,Support-Kerala,Support,Kerala,2001,25,2025-02-15,88185.88,Average,False,Pending,Invalid Phone,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3069,Nisha,Nathan,nisha.nathan@example.com,9156793341,Sales-Punjab,Sales,Punjab,1999,27,2029-05-31,125770.96,Excellent,False,On Leave,Invalid Join Date,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
